# Day 2: Data Cleaning + SQL Database Design (Bluestock Capstone)

This notebook demonstrates the data cleaning operations performed on the Mutual Fund raw CSV datasets and how they are loaded into a structured SQLite database (`bluestock_mf.db`) using SQLAlchemy.

In [1]:
import os
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Setup paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "notebooks" else Path("bluestock_mf_capstone")
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
db_path = project_root / "data" / "db" / "bluestock_mf.db"
schema_path = project_root / "sql" / "schema.sql"

print(f"Project root determined as: {project_root.resolve()}")

Project root determined as: C:\Users\Asus\OneDrive\Code_X\DAY 1 — Project Setup + Data Ingestion (ETL)\bluestock_mf_capstone


## 1. Clean `nav_history.csv`

We parse dates, drop duplicates, sort by `amfi_code` + `date`, and reindex to daily frequency to forward-fill (`ffill()`) weekend/holiday NAV records for each scheme. Finally, we filter out non-positive NAV values.

In [2]:
nav_path = raw_dir / "02_nav_history.csv"
df_nav = pd.read_csv(nav_path)
print(f"Original NAV History Shape: {df_nav.shape}")

df_nav["date"] = pd.to_datetime(df_nav["date"], errors="coerce")
df_nav = df_nav.dropna(subset=["date"])
df_nav = df_nav.drop_duplicates(subset=["amfi_code", "date"])

# Group by scheme and reindex to include all calendar dates
cleaned_nav_groups = []
for code, group in df_nav.groupby("amfi_code"):
    group = group.set_index("date").sort_index()
    all_dates = pd.date_range(start=group.index.min(), end=group.index.max(), freq="D")
    group = group.reindex(all_dates)
    group["amfi_code"] = code
    group["nav"] = group["nav"].ffill()
    group = group.reset_index().rename(columns={"index": "date"})
    cleaned_nav_groups.append(group)
    
df_nav_clean = pd.concat(cleaned_nav_groups, ignore_index=True)
df_nav_clean = df_nav_clean[df_nav_clean["nav"] > 0]

# Format date as string
df_nav_clean["date"] = df_nav_clean["date"].dt.strftime("%Y-%m-%d")
print(f"Cleaned NAV History Shape (including filled weekends): {df_nav_clean.shape}")
df_nav_clean.head(5)

Original NAV History Shape: (46000, 3)


Cleaned NAV History Shape (including filled weekends): (64320, 3)


,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


## 2. Clean `investor_transactions.csv`

We standardize `transaction_type` values to standard classifications (`SIP`, `Lumpsum`, `Redemption`), validate that investment amounts are strictly positive, normalize dates, and standardize `kyc_status` enums.

In [3]:
tx_path = raw_dir / "08_investor_transactions.csv"
df_tx = pd.read_csv(tx_path)
print(f"Original Transactions Shape: {df_tx.shape}")
print(f"Original transaction_types: {df_tx['transaction_type'].unique()}")

# Convert dates
df_tx["transaction_date"] = pd.to_datetime(df_tx["transaction_date"], errors="coerce")
df_tx = df_tx.dropna(subset=["transaction_date"])
df_tx["transaction_date"] = df_tx["transaction_date"].dt.strftime("%Y-%m-%d")

# Standardize type
type_map = {"sip": "SIP", "lumpsum": "Lumpsum", "redemption": "Redemption"}
df_tx["transaction_type"] = df_tx["transaction_type"].astype(str).str.strip().str.lower().map(type_map).fillna("Lumpsum")

# Validate amounts > 0
df_tx = df_tx[df_tx["amount_inr"] > 0]

# Mock units column if missing
if "units" not in df_tx.columns:
    df_tx["units"] = (df_tx["amount_inr"] / 100.0).round(4)
    
# Normalize KYC
kyc_map = {"verified": "Verified", "pending": "Pending", "failed": "Failed"}
df_tx["kyc_status"] = df_tx["kyc_status"].astype(str).str.strip().str.lower().map(kyc_map).fillna("Pending")

print(f"Cleaned Transactions Shape: {df_tx.shape}")
df_tx.head(3)

Original Transactions Shape: (32778, 13)
Original transaction_types: ['SIP' 'Redemption' 'Lumpsum']
Cleaned Transactions Shape: (32778, 14)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,units
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified,18.34
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified,3928.82
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified,9.12


## 3. Clean `scheme_performance.csv`

We validate numeric conversions and run a range constraint boundary check on the expense ratios (expected range: 0.1% to 2.5%).

In [4]:
perf_path = raw_dir / "07_scheme_performance.csv"
df_perf = pd.read_csv(perf_path)
print(f"Original Performance Shape: {df_perf.shape}")

# Convert performance columns to float
cols_to_numeric = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "expense_ratio_pct", "morningstar_rating"]
for col in cols_to_numeric:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors="coerce")

# Expense ratio boundary check (0.1% - 2.5%)
anomalies = df_perf[(df_perf["expense_ratio_pct"] < 0.1) | (df_perf["expense_ratio_pct"] > 2.5)]
print(f"Number of expense ratio anomalies detected: {len(anomalies)}")
if not anomalies.empty:
    print(anomalies[["amfi_code", "scheme_name", "expense_ratio_pct"]].head(5))
    
df_perf.head(3)

Original Performance Shape: (40, 19)
Number of expense ratio anomalies detected: 0


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High


## 4. SQLite Database Load and Row Count Verification

We initialize our SQLite connection, apply the relational schema, and load our tables. Finally, we execute simple queries to verify database counts match the original dataframes.

In [5]:
# Connect to database
engine = create_engine(f"sqlite:///{db_path}")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Query row counts from database
tables = ["dim_fund", "dim_date", "fact_nav", "fact_transactions", "fact_performance"]
print("Verify loaded SQLite row counts:")
for t in tables:
    try:
        cursor.execute(f"SELECT COUNT(*) FROM {t}")
        count = cursor.fetchone()[0]
        print(f"  - Table '{t}': {count} rows in database.")
    except Exception as e:
        print(f"  - Table '{t}': Error querying table: {e}")
        
conn.close()

Verify loaded SQLite row counts:
  - Table 'dim_fund': 40 rows in database.
  - Table 'dim_date': 1608 rows in database.
  - Table 'fact_nav': 64320 rows in database.
  - Table 'fact_transactions': 32778 rows in database.
  - Table 'fact_performance': 40 rows in database.
